# Bayesian Methods for ML — Module 1 Foundations

## Learning Objectives
- Understand Bayesian probability: prior, likelihood, and posterior
- Implement Bayes' theorem for parameter estimation
- Use PyMC3 for probabilistic modeling and Markov chain sampling
- Compare Bayesian vs frequentist approaches to inference

## Why This Matters
Skillsoft includes 2h 50m of Bayesian methods covering Bayesian concepts, PyMC3 computation, and advanced models (hierarchical linear models, Gaussian processes, mixture models). Bayesian methods are essential for uncertainty quantification, which is critical for ML Architect decisions about model risk and reliability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

try:
    import pymc as pm
    import arviz as az
    PYMC_AVAILABLE = True
    print(\"PyMC3/ArviZ available\")
except ImportError:
    PYMC_AVAILABLE = False
    print(\"PyMC3 not installed. Run: pip install pymc arviz\")
    # Fallback to scipy
    from scipy.special import beta as scipy_beta

sns.set_style(\"whitegrid\")
np.random.seed(42)
%matplotlib inline


## 1. Bayes' Theorem: From Concept to Computation


In [ ]:
# Bayes' theorem: P(H|E) = P(E|H) * P(H) / P(E)
# Example: Medical test screening
# P(disease) = 0.01 (1% prevalence)
# P(positive|disease) = 0.95 (sensitivity)
# P(positive|no disease) = 0.05 (false positive rate)

p_disease = 0.01
p_no_disease = 0.99
p_pos_given_disease = 0.95
p_pos_given_no_disease = 0.05

p_positive = p_pos_given_disease * p_disease + p_pos_given_no_disease * p_no_disease
p_disease_given_positive = (p_pos_given_disease * p_disease) / p_positive

print(f\"P(disease) = {p_disease:.2%}\")
print(f\"P(positive|disease) = {p_pos_given_disease:.2f}\")
print(f\"P(positive|no disease) = {p_pos_given_no_disease:.2f}\")
print(f\"P(positive) = {p_positive:.4f}\")
print(f\"P(disease|positive) = {p_disease_given_positive:.4f}\")
",print(f\"Despite positive test, probability of disease is only {p_disease_given_positive:.1%}!\")
",
# Visualize: Bayesian updating as odds ratio
odds_prior = p_disease / p_no_disease
likelihood_ratio = p_pos_given_disease / p_pos_given_no_disease
odds_posterior = odds_prior * likelihood_ratio
p_posterior = odds_posterior / (1 + odds_posterior)

print(f\"\
Odds form:\")
f\"  Prior odds: {odds_prior:.3f}\",
    f\"LR: {likelihood_ratio:.2f}\",
    f\"Posterior odds: {odds_posterior:.3f}\",
    f\"Posterior probability: {p_posterior:.4f}\",
]

## 2. Bayesian Inference: Coin Flip Revisited (MLE vs MAP vs Full Posterior)


In [ ]:
# If PyMC is available, use it; otherwise use analytic Beta conjugate
if PYMC_AVAILABLE:
    # PyMC3 approach
    heads, total = 7, 10

    with pm.Model() as model:
        # Prior: θ ~ Beta(1, 1) (uniform)
        theta = pm.Beta('theta', alpha=1, beta=1)
        # Likelihood: y ~ Bernoulli(θ)
        y = pm.Bernoulli('y', p=theta, observed=np.ones(total))
        
        # Sample from posterior
        trace = pm.sample(1000, tune=1000, return_inferencedata=False)
        
        print(f\"PyMC3 MAP: {pm.find_map(model)['theta']:.4f}\")
        print(f\"PyMC3 Posterior mean: {trace['theta'].mean():.4f}\")
        print(f\"PyMC3 Posterior std: {trace['theta'].std():.4f}\")
        
        az.plot_trace(trace, var_names=['theta'])
        plt.show()

    # Compare to analytic MAP
    def map_bernoulli(heads, total, alpha=1, beta=1):
    return (heads + alpha) / (total + alpha + beta)
    
    mle = heads / total
    map_analytic = map_bernoulli(heads, total, alpha=1, beta=1)
    print(f\"\
Analytic MAP: {map_analytic:.4f} (same as PyMC3 MAP with Beta(1,1) prior)\")
else:
    # Analytic solution with Beta conjugate prior
    heads, total = 7, 10
    
    # Prior: Beta(alpha, beta)
    # Posterior: Beta(alpha + heads, beta + tails)
    
    # Uniform prior Beta(1, 1)
    mle = heads / total
    map_uniform = (heads + 1) / (total + 2)
    
    # Beta(2, 2) prior: favors 0.5, stronger belief
    map_beta22 = (heads + 2) / (total + 4)
    
    # Beta(5, 5) prior: strong belief toward 0.5
    map_beta55 = (heads + 5) / (total + 10)
    
    print(f\"Data: {heads} heads in {total} flips\")
    print(f\"MLE: θ = {mle:.4f} ({heads/total*100:.1f}%)\")
    print(f\"MAP (uniform prior Beta(1,1)): θ = {map_uniform:.4f}\")
    print(f\"MAP (weak prior Beta(2,2)): θ = {map_beta22:.4f}\")
    print(f\"MAP (strong prior Beta(5,5)): θ = {map_beta55:.4f}\")
    
    # Posterior distributions
    import numpy as onp
    # Sample from posterior analytically
    alpha_post = 1 + heads  # uniform prior
    beta_post = 1 + (total - heads)
    
    posterior_samples = np.random.beta(alpha_post, beta_post, size=10000)
    
    print(f\"\
Uniform prior Beta(1,1) posterior:\")
    print(f\"  Mean: {posterior_samples.mean():.4f}\")
    print(f\"  95% credible interval: [{np.percentile(posterior_samples, 2.5):.4f}, {np.percentile(posterior_samples, 97.5):.4f}]\")
    
    # With strong prior
    alpha_post5 = 5 + heads
    beta_post5 = 5 + (total - heads)
    posterior_samples5 = np.random.beta(alpha_post5, beta_post5, size=10000)
    
    print(f\"\
Strong prior Beta(5,5) posterior:\")
    print(f\"  Mean: {posterior_samples5.mean():.4f}\")
    print(f\"  95% credible interval: [{np.percentile(posterior_samples5, 2.5):.4f}, {np.percentile(posterior_samples5, 97.5):.4f}]\")


## 3. PyMC3: Linear Regression with Uncertainty


In [ ]:
if PYMC_AVAILABLE:
    # Generate synthetic data with noise
    np.random.seed(42)
    n = 50
    X = np.linspace(0, 10, n)
    y_true = 3.0 * X + 2.0  # y = 3x + 2
    y = y_true + np.random.normal(scale=2.0, size=n)  # Add noise

    with pm.Model() as model:
        # Priors for unknown parameters
        alpha = pm.Normal('alpha', mu=0, sigma=10)  # Intercept
        beta = pm.Normal('beta', mu=0, sigma=10)  # Slope
        sigma = pm.HalfNormal('sigma', sigma=1)  # Noise scale

# Expected value of y for each given X
",        mu = alpha + beta * X

# Likelihood (sampling distribution) observed data ~ Normal
",        y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)

# Infer posterior
",        trace = pm.sample(2000, tune=1000, cores=1, random_seed=123)

    # Summary of posterior
    summary = az.summary(trace, var_names=['alpha', 'beta', 'sigma'])
    print(\"\
Posterior summary:\")
    print(summary)

    # Posterior predictive checks
    ppc = pm.sample_posterior_predictive(trace, samples=500)

    # Plot: data + posterior lines
    plt.figure(figsize=(10, 6))
    plt.plot(X, y, 'k.', label='Observed data')
    # Plot 5 random posterior predictive lines
    for i in range(5):
        idx = np.random.choice(len(trace['alpha']))
        y_pred = trace['alpha'][idx] + trace['beta'][idx] * X
        plt.plot(X, y_pred, 'r-', alpha=0.3)
    plt.plot(X, y_true, 'b--', label='True function (y=3x+2)')
    plt.xlabel('X')
    plt.ylabel('y')
    plt.title('Bayesian Linear Regression Posterior Predictive')
    plt.legend()
    plt.show()


## 4. Exercises


### Exercise 1: Bayesian A/B Test

In an A/B test, variant A has 50 conversions out of 1000 visitors, and variant B has 60 conversions out of 1100 visitors.
Use Bayesian methods to compute the probability that B > A, and report a 95% credible interval for the difference in conversion rates.


### Exercise 2: Parameter Estimation with Different Priors

Given 15 heads in 20 coin flips, compare MLE, MAP with Beta(1,1), Beta(2,2), and Beta(5,5) priors. Plot the three posterior distributions on the same axes.


### Exercise 3: Bayesian Logistic Regression (Conceptual)

Describe how you would modify the Bayesian linear regression notebook to perform Bayesian logistic regression for classification. What changes to the likelihood function? How would you interpret the posterior?

## 5. Summary

| Concept | Skillsoft Course | Notebook |
|---------|-----------------|----------|
| Bayes' theorem | Bayesian Concepts & Core Components (1h) | 06_bayesian_methods.ipynb |
| Parameter estimation (MLE/MAP) | Implementing Bayesian Model & Computation with PyMC (47m) | 06_bayesian_methods.ipynb |
| Bayesian linear regression | Advanced Bayesian Computation Model (51m) | 06_bayesian_methods.ipynb |
| PyMC3 MCMC sampling | Implementing Bayesian Model & Computation with PyMC | 06_bayesian_methods.ipynb |
| Uncertainty quantification | Key for ML Architect risk assessment | 06_bayesian_methods.ipynb |

Key Takeaway: Bayesian methods give you not just point estimates but full probability distributions over parameters — essential for risk-aware ML decisions as an Architect.